# 3 N-BEATS: putting the decomposition *inside* the architecture

Suppose you computed a trend slope and a set of FFT band energies by hand and bolted them onto
the model as a second input stream. Measured, the model ignores them — and the diagnosis is that
a linear layer over the raw window can already compute a trend slope, since it is a weighted
sum. You handed it a column it already had.

That diagnosis suggests an obvious next move. If trend and seasonality are the right vocabulary
for a timeseries, do not compute them outside the model and pass them in. **Make them the shape
the model is allowed to predict in.**

That is N-BEATS (Oreshkin et al., 2020). Instead of a network that outputs 24 free numbers, you
get a network that outputs *coefficients of a basis* — a polynomial for trend, a Fourier series
for seasonality — and the forecast is whatever those coefficients build. The architecture cannot
predict an arbitrary squiggle. It can only predict a trend plus a seasonal pattern, and it has to
say how much of each.

**You cannot download this one.** There is no pretrained N-BEATS the way there is a pretrained
`resnet18`, and the reason is structural rather than an oversight: N-BEATS is fitted per series
and per horizon, so there is no transferable feature extractor to ship. (`neuralforecast` has an
implementation, but it depends on pytorch-lightning *and* pyspark, and its top-level model wants
a Lightning batch dict.) So we build it — about forty lines, mirroring
[`neuralforecast/models/nbeats.py`](https://github.com/Nixtla/neuralforecast/blob/main/neuralforecast/models/nbeats.py),
close enough that you can read theirs afterwards and recognise every part.

In [ ]:
import sys
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from loguru import logger
from notebooktester import param
from solar_10_minutes import load_solar
from sunspot_daily import load_sunspot_daily
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components.metrics import MAE
from vectormesh.training import Trainer, TrainerSettings
from windowing import window_index

W, H = 288, 144  # two days of history, one day of horizon, at 10-minute resolution
# one day, in 10-minute steps -- the seasonality we believe is in this data
PERIOD = 144
HARMONICS = 10  # enough to fit the sharp night/day edge -- see 3.2
SEEDS = (0, 1, 2)
# a ceiling, not a choice: early stopping ends each run where its own validation loss
# stops falling (3.4), and a run that reaches the ceiling shows it as stopped = 40
MAX_EPOCHS = param(40, test=1)
# the N-BEATS validation loss bumps up and down from epoch to epoch by about as much as it
# still improves (see the loss curves in 3.4), so a stop needs five epochs without a new best,
# not one bump
PATIENCE = 5
logger.remove()  # Trainer logs every epoch; keep warnings only
logger.add(sys.stderr, level="WARNING")
torch.manual_seed(0)

## 3.1 A series with something for a Fourier basis to find

A seasonal basis is only worth having if there is seasonality. Not every series has
some at a period you can window over — `3.6` comes back to that question directly.

So this notebook opens on **solar power production** — ten-minute readings from 137
sites in the Monash `solar_10_minutes` set, where the sun's daily arc is most of the
story: every site sits at exactly zero for roughly half of each day, night after
night, whatever the weather is doing to the daytime hump.

In [ ]:
def split_scale_window(series: list[torch.Tensor], stride: int = 6) -> dict:
    # 60/20/20 in time, per series, scaled on that series' own training span, then windowed
    parts = {"train": [[], []], "val": [[], []], "test": [[], []]}
    for s in series:
        n = len(s)
        cuts = {
            "train": (0, int(0.6 * n)),
            "val": (int(0.6 * n), int(0.8 * n)),
            "test": (int(0.8 * n), n),
        }
        mu = s[: cuts["train"][1]].mean()
        sigma = s[: cuts["train"][1]].std().clamp(min=1e-6)
        for split, (a, b) in cuts.items():
            v = (s[a:b] - mu) / sigma
            win = v[window_index(len(v), W + H, stride)]
            parts[split][0].append(win[:, :W])
            parts[split][1].append(win[:, W:])
    return {k: (torch.cat(v[0]), torch.cat(v[1])) for k, v in parts.items()}


# 20 of 137 sites, pooled -- enough windows that a linear layer is not data-starved (3.4
# needs that to be true, or a win there would prove nothing about the architecture).
power = split_scale_window(load_solar(W, H, n_series=20))
for split, (x, y) in power.items():
    print(f"{split:5s} x {tuple(x.shape)}  y {tuple(y.shape)}")

In [ ]:
train_x = power["train"][0]
sample = train_x[0]

n = len(sample)
z = (sample - sample.mean()) * torch.hann_window(n, periodic=False)
spec = torch.fft.rfft(z).abs() ** 2
steps = 1.0 / torch.fft.rfftfreq(n, d=1.0)[1:]
hours = steps * (10 / 60)  # 10-minute steps -> hours, for a readable axis

fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].plot(sample, lw=0.8)
ax[0].set_title("one 2-day window of solar power (288 ten-minute steps)")
ax[0].set_xlabel("10-minute steps")
ax[1].loglog(hours, spec[1:], lw=0.8)
ax[1].axvline(24, color="crimson", ls="--", lw=1)
ax[1].text(24, spec[1:].max(), " 24h", color="crimson", va="top", fontsize=9)
ax[1].set_xlabel("period (hours)")
ax[1].set_title("its spectrum")
plt.tight_layout()

A spike at 24 hours you can see without squinting. That is the thing a Fourier basis is for.

## 3.2 A basis is a fixed set of shapes; the network picks the mix

Every N-BEATS block does the same two steps:

1. an ordinary MLP maps the input window to a short vector of coefficients, **theta**;
2. a **basis** turns theta into two curves — a *backcast* (its reconstruction of the input) and a
   *forecast* (its prediction of the future).

The basis has **no learnable parameters**. It is a fixed matrix of shapes, and all the network
does is decide how much of each shape to use. Swap the basis and you change what the block is
*able* to say.

The trend basis is powers of normalised time: a constant, a ramp, a curve, a bend.

In [ ]:
class TrendBasis(nn.Module):
    # theta -> a low-order polynomial in normalised time, for backcast and forecast

    backcast_basis: torch.Tensor
    forecast_basis: torch.Tensor

    def __init__(self, degree: int, backcast_size: int, forecast_size: int):
        super().__init__()
        self.size = degree + 1
        powers = lambda n: torch.stack(
            [torch.linspace(0, 1, n) ** p for p in range(self.size)]
        )
        self.register_buffer("backcast_basis", powers(backcast_size))  # (size, W)
        self.register_buffer("forecast_basis", powers(forecast_size))  # (size, H)

    def forward(self, theta):
        backcast = torch.einsum("bp,pt->bt", theta[:, : self.size], self.backcast_basis)
        forecast = torch.einsum("bp,pt->bt", theta[:, self.size :], self.forecast_basis)
        return backcast, forecast


trend = TrendBasis(degree=3, backcast_size=W, forecast_size=H)
fig, ax = plt.subplots(figsize=(9, 2.5))
for p, row in enumerate(trend.forecast_basis):
    ax.plot(row, label=f"t^{p}")
ax.set_title("the trend basis: four shapes, no parameters")
ax.legend(fontsize=8, ncol=4)
plt.tight_layout()
print(
    f"theta for this basis has {2 * trend.size} numbers "
    f"({trend.size} for the backcast, {trend.size} for the forecast)"
)

And the seasonal basis is a Fourier series: sines and cosines at harmonics of a period you name.

Look at what that means. Computing FFT *band energies* and handing them to the model as inputs
puts the spectrum on the input side. Here the same sines and cosines are the **output** vocabulary — the model
does not receive a spectrum, it predicts *in* one. Same mathematics, opposite direction.

In [ ]:
class SeasonalityBasis(nn.Module):
    # theta -> a Fourier series at harmonics of a known period

    backcast_basis: torch.Tensor
    forecast_basis: torch.Tensor

    def __init__(
        self, harmonics: int, period: int, backcast_size: int, forecast_size: int
    ):
        super().__init__()
        self.size = 2 * harmonics
        # the backcast grid runs up to the forecast origin, the forecast grid starts at it,
        # so the two are one continuous timeline and the phase carries across the boundary
        for name, n, offset in (
            ("backcast", backcast_size, -backcast_size),
            ("forecast", forecast_size, 0),
        ):
            t = (torch.arange(n, dtype=torch.float32) + offset) / period
            rows = [
                f(2 * np.pi * k * t)
                for k in range(1, harmonics + 1)
                for f in (torch.cos, torch.sin)
            ]
            self.register_buffer(f"{name}_basis", torch.stack(rows))

    def forward(self, theta):
        backcast = torch.einsum("bp,pt->bt", theta[:, : self.size], self.backcast_basis)
        forecast = torch.einsum("bp,pt->bt", theta[:, self.size :], self.forecast_basis)
        return backcast, forecast


seasonal = SeasonalityBasis(
    harmonics=HARMONICS, period=PERIOD, backcast_size=W, forecast_size=H
)

fig, ax = plt.subplots(figsize=(9, 2.5))
for i in range(4):
    ax.plot(seasonal.forecast_basis[i], lw=1)
ax.set_title(
    f"the seasonality basis: harmonics of a {PERIOD}-step (24h) period (first four shown)"
)
plt.tight_layout()

# a basis is only trustworthy if it can express the thing it claims to
target = torch.cos(2 * np.pi * torch.arange(H, dtype=torch.float32) / PERIOD)
theta = torch.zeros(1, 2 * seasonal.size)
theta[0, seasonal.size] = 1.0  # one unit of the first cosine, nothing else
_, reconstructed = seasonal(theta)
print(
    f"reconstructs a pure {PERIOD}-step cosine exactly: "
    f"max error {float((reconstructed[0] - target).abs().max()):.2e}"
)

## 3.3 The block, and the stacking that makes it work

A block is the MLP plus a basis. The interesting part is what happens *between* blocks.

Each block emits a backcast — its own account of the input — and that account is **subtracted from
the input before the next block sees it**. Block two therefore works on what block one could not
explain. Meanwhile the forecasts are **added up**. Oreshkin et al. call this *doubly residual
stacking*, and it is the whole mechanism: subtract backwards, accumulate forwards.

Give the first block a trend basis and the second a seasonal one, and the model is forced to
explain the trend first and treat what is left as seasonal. The decomposition is not something you
compute; it is something the architecture cannot avoid doing.

In [ ]:
class IdentityBasis(nn.Module):
    # theta IS the curve - no structure imposed. This is "generic" N-BEATS.

    def __init__(self, backcast_size: int, forecast_size: int):
        super().__init__()
        self.size = backcast_size

    def forward(self, theta):
        return theta[:, : self.size], theta[:, self.size :]


class NBeatsBlock(nn.Module):
    def __init__(
        self,
        basis,
        input_size: int,
        theta_size: int,
        hidden: int = 256,
        layers: int = 3,
    ):
        super().__init__()
        mlp, d = [], input_size
        for _ in range(layers):
            mlp += [nn.Linear(d, hidden), nn.ReLU()]
            d = hidden
        self.mlp = nn.Sequential(*mlp)
        self.theta = nn.Linear(hidden, theta_size)
        self.basis = basis

    def forward(self, x):
        return self.basis(self.theta(self.mlp(x)))


class NBeats(nn.Module):
    def __init__(self, blocks: list[NBeatsBlock]):
        super().__init__()
        self.blocks = nn.ModuleList(blocks)

    def forward(self, x, per_block: bool = False):
        residual = x
        forecast = x[:, -1:].expand(-1, H).clone()  # start at the naive forecast
        contributions = []
        for block in self.blocks:
            backcast, block_forecast = block(residual)
            residual = residual - backcast  # <- subtract what was explained
            forecast = forecast + block_forecast  # <- accumulate what was predicted
            contributions.append(block_forecast)
        return (forecast, contributions) if per_block else forecast


def interpretable(degree=3, harmonics=HARMONICS, period=PERIOD, hidden=256):
    t = TrendBasis(degree, W, H)
    s = SeasonalityBasis(harmonics, period, W, H)
    return NBeats(
        [NBeatsBlock(t, W, 2 * t.size, hidden), NBeatsBlock(s, W, 2 * s.size, hidden)]
    )


def generic(n_blocks=2, hidden=256):
    return NBeats(
        [NBeatsBlock(IdentityBasis(W, H), W, W + H, hidden) for _ in range(n_blocks)]
    )


model = interpretable()
out, parts = model(train_x[:4], per_block=True)
print(
    f"forecast {tuple(out.shape)}, from {len(parts)} stacks of {tuple(parts[0].shape)}"
)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 3.4 Baselines first

The free baselines first, then the cheap trained model, then the expensive one. With a real
24-hour cycle, seasonal naive should be the stronger of the two free baselines. Check that
before going further; 3.6 runs the same two baselines on a series without a cycle at this period.

**How long each model trains.** No epoch count is picked here. Every model trains with early
stopping, as in lesson 1 (`1_representation_and_head/03_training`): after each epoch the
`Trainer` computes the loss on the validation split, keeps the weights of the epoch with the
lowest one, and stops once `PATIENCE` epochs in a row bring no new lowest. A linear layer and a
three-layer MLP stack converge at different speeds, so each one stops where its own validation
loss says. That leaves two jobs for three splits: **validation** decides when to stop, and
**test**, which neither training nor stopping ever sees, is the split every score in this
notebook is reported on.

Before the next cell runs, write down two expectations: will the N-BEATS models beat the linear
layer, and will any trained model beat seasonal naive, which only copies yesterday?

In [ ]:
def mae(y, yhat):
    return float((y - yhat).abs().mean())


def fit(make_model, data, seeds=SEEDS, max_epochs=MAX_EPOCHS, lr=1e-3, batch_size=128):
    # One Trainer per seed. Early stopping watches the validation loss (MSE) and hands back
    # the model of its best epoch; test is scored by hand afterwards on that model, because
    # Trainer never sees the test split.
    xtr, ytr = data["train"]
    xva, yva = data["val"]
    xte, yte = data["test"]
    validloader = DataLoader(TensorDataset(xva, yva), batch_size=len(xva))
    runs = {"test": [], "val": [], "best": [], "stopped": [], "secs": [], "curves": []}
    for seed in seeds:
        torch.manual_seed(seed)
        model = make_model()
        trainloader = DataLoader(
            TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
        )
        settings = TrainerSettings(
            epochs=max_epochs,
            metrics=[MAE()],
            logdir=Path("modellogs").resolve(),
            train_steps=len(trainloader),
            valid_steps=len(validloader),
            optimizer_kwargs={"lr": lr},
            earlystop_kwargs={"patience": PATIENCE, "save": False, "verbose": False},
        )
        curve = []  # (train loss, validation loss) per epoch, for the loss curves

        def record(epoch, train_loss, valid_loss, metrics, curve=curve):
            curve.append((train_loss, valid_loss))

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=nn.MSELoss(),
            optimizer=torch.optim.Adam,
            traindataloader=trainloader,
            validdataloader=validloader,
            reporters=[record],
            progress=False,
        )
        t0 = time.time()
        result = trainer.loop()  # trainer.model now holds the weights of the best epoch
        runs["secs"].append(time.time() - t0)
        runs["val"].append(result.metric_dict["MAE"])
        runs["best"].append(result.epoch + 1)  # Trainer counts epochs from 0
        runs["stopped"].append(result.stopped_epoch + 1)
        runs["curves"].append(curve)
        model.eval()
        with torch.no_grad():
            runs["test"].append(mae(yte, model(xte)))
    return {
        "mae": float(np.mean(runs["test"])),
        "std": float(np.std(runs["test"])),
        "val": float(np.mean(runs["val"])),
        "best": float(np.mean(runs["best"])),
        "stopped": float(np.mean(runs["stopped"])),
        "secs": float(np.mean(runs["secs"])),
        "params": sum(p.numel() for p in model.parameters()),
        "curves": runs["curves"],
        "model": model,
    }


class Linear(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(W, H)

    def forward(self, x):
        return self.fc(x)


def free(forecast, data) -> dict[str, Any]:
    # a baseline with nothing to train: no seeds, no epochs, no parameters
    (xva, yva), (xte, yte) = data["val"], data["test"]
    nan = float("nan")
    return {
        "mae": mae(yte, forecast(xte)),
        "std": 0.0,
        "val": mae(yva, forecast(xva)),
        "best": nan,
        "stopped": nan,
        "secs": 0.0,
        "params": 0,
    }


def naive(x):
    return x[:, -1:].expand(-1, H)


def seasonal_naive(x):
    return x[:, -PERIOD:][:, :H]


results: dict[str, dict[str, Any]] = {
    "naive (repeat last)": free(naive, power),
    "seasonal naive (24h)": free(seasonal_naive, power),
}
for name, r in results.items():
    print(f"{name:24s} test MAE {r['mae']:.4f}   val MAE {r['val']:.4f}")
xte, yte = power["test"]

In [ ]:
for name, make in (
    ("linear", Linear),
    ("N-BEATS generic", generic),
    ("N-BEATS interpretable", interpretable),
):
    r = fit(make, power)
    results[name] = r
    print(
        f"{name:24s} test MAE {r['mae']:.4f} +/- {r['std']:.4f}   val MAE {r['val']:.4f}   "
        f"best epoch {r['best']:4.1f}   stopped {r['stopped']:4.1f}   {r['secs']:5.1f}s"
    )

In [ ]:
def table(results):
    head = f"{'model':24s} {'test MAE':>9} {'+/-':>7} {'val MAE':>8} {'best ep':>8} {'stopped':>8} {'params':>9} {'secs':>7}"
    print(head)
    print("-" * len(head))
    for name in sorted(results, key=lambda k: results[k]["mae"]):
        r = results[name]
        ep = lambda v: f"{v:8.1f}" if v == v else f"{'-':>8}"
        print(
            f"{name:24s} {r['mae']:9.4f} {r['std']:7.4f} {r['val']:8.4f} {ep(r['best'])} {ep(r['stopped'])} "
            f"{r['params']:9,} {r['secs']:7.1f}"
        )


table(results)

In [ ]:
def plot_curves(results, names, title):
    fig, axes = plt.subplots(1, len(names), figsize=(4.5 * len(names), 3), sharey=True)
    for ax, name in zip(axes, names):
        for seed, curve in zip(SEEDS, results[name]["curves"]):
            train, valid = np.array(curve).T
            epochs = np.arange(1, len(curve) + 1)
            (line,) = ax.plot(epochs, valid, lw=1.5, label=f"seed {seed} validation")
            ax.plot(epochs, train, lw=0.8, ls=":", color=line.get_color())
            best = int(np.argmin(valid))
            ax.plot(epochs[best], valid[best], "o", color=line.get_color())
        ax.set_title(name)
        ax.set_xlabel("epoch")
    axes[0].set_ylabel("MSE loss (scaled units)")
    axes[-1].legend(fontsize=7)
    fig.suptitle(title)
    plt.tight_layout()


trained = ["linear", "N-BEATS generic", "N-BEATS interpretable"]
plot_curves(
    results,
    trained,
    "validation loss per epoch (solid), training loss (dotted); the dot is the epoch kept",
)

**Read the test column; validation is where each run chose to stop.**

**N-BEATS beats the linear layer, by far more than the seed noise.** On test, generic N-BEATS
scores 0.2508 ± 0.0011 and linear 0.3022 ± 0.0051: a gap of 0.051, ten times the larger of the
two seed stds. The interpretable variant lands at 0.2571 ± 0.0057, level with generic (the gap
of 0.006 is about one seed std). Validation gives the same order (0.2246 and 0.2239 against
0.2502).

**Each model trained as long as its own validation loss asked.** Linear kept its weights of
epoch 6.3 on average and stopped at 11.3; generic kept epoch 3.7 and stopped at 8.7;
interpretable kept 5.3 and stopped at 10.3. The loss curves show why so early: the validation
loss of all three is flat from the second or third epoch on, while the training loss of both
N-BEATS models keeps falling. More epochs fit the training months better and the validation
months no better. So none of the three was still learning when the comparison was made.

**No trained model beats seasonal naive on test.** Copying yesterday scores 0.2201; the best
trained model, 0.2508. On validation the two N-BEATS models edge just past it (0.2246 and
0.2239 against 0.2274), so the two splits disagree. Two things differ between them. First,
validation picked each trained model's epoch, so its validation score is optimistic, while
seasonal naive had nothing to pick. Second, the 60/20/20 cut is in time, over one calendar
year: training covers January to early August, validation early August to mid October, test
mid October to December. From validation to test, seasonal naive gets slightly better
(0.2274 → 0.2201) and every trained model gets worse (generic 0.2246 → 0.2508, linear
0.2502 → 0.3022). The validation curves lie in a narrow band across epochs, so picking the
lowest one buys little; the likelier reading is that the trained models meet months further
from what they were trained on, while copying yesterday works in any month. This notebook
does not test that reading; E1 does.

**Where the gap over linear comes from.** `nn.Linear(288, 144)` has one free weight per
(input step, output step) pair, 41,616 numbers, and the notebook has 103,700 training windows,
so the linear layer is not short of data. Generic N-BEATS differs from it in more than one
factor: it is deeper (three hidden layers of 256 per block, two blocks), has 633,184
parameters, and subtracts each block's backcast before the next block. It has no seasonal
basis at all, and it scores as well as the interpretable model, which does. So this table does
not credit the gap to the Fourier basis: what the basis bought on solar power is the
decomposition you can read off in 3.5, at a score level with the unconstrained stack.

Hold that thought until 3.6, where the series changes but the architecture does not.

## 3.5 The payoff: ask the model what it thinks the series is made of

The interpretable model does not just score well here — it can also do something
none of the others in this lesson can. `per_block=True` returns each stack's
contribution separately, so you can plot the trend it found and the seasonality it
found as **two curves that sum to the forecast**.

This is not a visualisation trick bolted on afterwards. The stacks are separate because
the architecture made them separate.

In [ ]:
interp = results["N-BEATS interpretable"]["model"]
interp.eval()
with torch.no_grad():
    forecast, (trend_part, seasonal_part) = interp(xte[:512], per_block=True)

picks = [3, 61, 140]
fig, axes = plt.subplots(1, len(picks), figsize=(14, 3.2), sharey=True)
for ax, i in zip(axes, picks):
    ax.plot(range(-PERIOD, 0), xte[i, -PERIOD:], color="0.6", lw=1, label="history")
    ax.plot(range(H), yte[i], color="k", lw=1.5, label="truth")
    ax.plot(range(H), forecast[i], color="crimson", lw=2, ls="--", label="forecast")
    ax.plot(
        range(H), trend_part[i], color="tab:blue", lw=1, ls="--", label="trend stack"
    )
    ax.plot(
        range(H),
        seasonal_part[i],
        color="tab:green",
        lw=2,
        ls="--",
        label="seasonality stack",
    )
    ax.axvline(0, color="0.85")
axes[0].set_ylabel("scaled solar power")
axes[-1].legend(fontsize=7)
fig.suptitle("the forecast, and the two things it is made of")
plt.tight_layout()

ratio = float(seasonal_part.abs().mean() / trend_part.abs().mean())
print(f"mean |trend contribution|       {float(trend_part.abs().mean()):.4f}")
print(f"mean |seasonality contribution| {float(seasonal_part.abs().mean()):.4f}")
print(f"ratio seasonality / trend       {ratio:.3f}")

The two stacks carry **comparable weight** (ratio 0.946) — the trend stack does
slightly more work than the seasonal one, which makes sense for a series whose
daytime peak height drifts with the weather day to day, on top of a shape that
repeats every 24 hours. The model has been forced to state, in numbers you can read,
how much of its forecast is "today looks generally like this stretch of days" and how
much is "it is 2pm".

On this series the interpretable model does not trade accuracy for that legibility: it
beat linear and is level with generic (3.4). That is not guaranteed to happen together;
3.6 checks whether it still does once the series' own structure stops cooperating.

Keep this ratio. It is about to change.

## 3.6 The same architecture on a series with no short season

Daily sunspot counts have the opposite of solar power's spectrum: `acf(1d)` near 1,
`acf(27d)` real but weak (the Sun's ~27-day rotation, see `sunspot_daily.py`), and the
single largest peak in the whole spectrum sitting at the ~11-year solar cycle — far too
slow for any window this notebook could hold. The architecture built above
has a seasonality stack tuned to a 144-step period. There is nothing at 144 days for
it to find, and the thing that *is* real here (27 days) is not what it was tuned to
either.

Run it unchanged, at the same `W=288, H=144` solar power was windowed at, and watch
the ratio.

In [ ]:
sun = load_sunspot_daily()
mu, sigma = (
    sun.values[sun.splits["train"]].mean(),
    sun.values[sun.splits["train"]].std(),
)
sunspot_windows = {}
for split in ("train", "val", "test"):
    values = (sun.values[sun.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, 6)]
    sunspot_windows[split] = (win[:, :W], win[:, W:])
print({k: len(v[0]) for k, v in sunspot_windows.items()})

sun_xte, sun_yte = sunspot_windows["test"]
sun_results = {
    "naive (repeat last)": free(naive, sunspot_windows),
    "seasonal naive (144d)": free(seasonal_naive, sunspot_windows),
}
for name, make in (
    ("linear", Linear),
    ("N-BEATS generic", generic),
    ("N-BEATS interpretable", interpretable),
):
    r = fit(make, sunspot_windows)
    sun_results[name] = r
    if name == "N-BEATS interpretable":
        r["model"].eval()
        with torch.no_grad():
            _, (t_part, s_part) = r["model"](sun_xte[:512], per_block=True)
        sun_ratio = float(s_part.abs().mean() / t_part.abs().mean())

table(sun_results)
print(f"\nratio seasonality / trend on sunspot:      {sun_ratio:.3f}")
print(f"                    on solar power:      {ratio:.3f}")

In [ ]:
plot_curves(
    sun_results,
    trained,
    "sunspot: validation loss per epoch (solid), training loss (dotted); the dot is the epoch kept",
)

Two things move, and neither is a simple flip.

**The seasonality stack goes quiet.** Its contribution relative to the trend stack drops from
0.946 (solar power, roughly balanced) to 0.220 (sunspot, the trend stack now doing more than
four times the work), because there is nothing at a 144-day period for it to spend itself on,
and not the real 27-day rotation either, since the stack was never tuned to it. The
decomposition is still produced (the architecture has no choice), but it has become much less
informative. A stack tuned to a period the data does not have raises no error; it is a
component that quietly contributes less, the same failure mode a designed feature stream hits
when the model ends up ignoring it.

**The seasonal basis loses its place; the deeper stack keeps its lead.** On sunspot's test
split, generic N-BEATS scores 0.2912 ± 0.0015 against linear's 0.2999 ± 0.0002: a gap of
0.0087, about six times generic's seed std, smaller than on solar but still a result. The
interpretable model scores 0.2982 ± 0.0058: its gap to linear (0.0017) is smaller than its own
seed std, so it ties linear. On solar power it was level with generic; here it falls back to
the linear layer. The one thing that separates the two N-BEATS models is the basis, and the
one thing that changed between the two series is whether the 144-step period means anything.
The stopping epochs in the table show that this is not a budget effect either: each model
again stopped where its own validation loss flattened.

**Headroom over naive is not the same size either.** On solar power, copying yesterday alone
cuts the naive baseline's error by 70% (0.220 against 0.737), and no trained model did better
than that copy on test. On sunspot, seasonal naive is worse than plain naive, and the best
model cuts naive's error by 21% (0.291 against 0.367). The autocorrelation says why: sunspot's
`acf(1d)` sits at 0.96 (`sunspot_daily.py`), so "repeat yesterday" is already most of the
answer, and no architecture closes much of a gap that small.

So the axes are independent, and conflating them is the mistake this section exists to
prevent:

| | seasonality at this window's period | headroom over naive | what the test split shows |
|---|---|---|---|
| solar power | strong, near-deterministic (24h) | large: seasonal naive alone cuts 70% | both N-BEATS beat linear (10x the noise), interpretable level with generic; none beats seasonal naive |
| sunspot | absent at 144d (real period is 27d, and 11yr besides) | small: best model cuts 21% | generic beats linear (6x its noise); interpretable ties linear |

**A basis is a bet about the data.** Where its period is real, it bought a readable
decomposition at no measurable cost in score; where it is not, it gave up the lead the
unconstrained stack still had. That is the concrete form of "no free lunch" this notebook has
been building toward: *name the period you are betting on, check whether the series actually
has it, and expect the bet to pay only when it does.*

## 3.7 A cycle that is not a season: sunspots

You have already met this series. MADS-DAV `03.2-statistics-of-time.ipynb` ran
`statsmodels.seasonal_decompose` on yearly sunspot counts and watched it fail: the
trend absorbed a cycle "of more than 100 years", the residual kept "very distinct
sine-wave patterns", and the notebook's own closing suggestions named "finding a way
to model the very long (+100 year) cycle" as unfinished business — handed forward on
purpose: *"In the course on Machine Learning we will go much deeper into this topic."*

This is that hand-off, with the basis classes you just built rather than a black-box
decomposition. Three named solar cycles sit in this record, at very different
timescales — the Schwabe cycle (~11yr), the Gleissberg cycle (~87yr, an amplitude
modulation of Schwabe), and the Suess/de Vries cycle (~210yr). The question is not
"does a fixed-period basis work here" — you already know a period must be roughly
fixed for that (3.6). It is: **how many of these three periods can 203 years of data
actually support?**


In [ ]:
def load_sunspot() -> torch.Tensor:
    # same rolling-eval convention as every Monash config: "test" holds the longest row.
    ds = load_dataset(
        "Monash-University/monash_tsf",
        revision="refs/convert/parquet",
        data_files={"test": "sunspot/test/0000.parquet"},
    )["test"][0]
    values = np.asarray(ds["target"], dtype="float32")
    dates = pd.date_range(start=ds["start"], periods=len(values), freq="D")
    yearly = pd.Series(values, index=dates).resample("YE").mean().dropna()
    return torch.tensor(yearly.to_numpy(), dtype=torch.float32), yearly.index


sunspot_yearly, sunspot_years = load_sunspot()
sun = (sunspot_yearly - sunspot_yearly.mean()) / sunspot_yearly.std()
N = len(sun)
print(
    f"{N} yearly means, {sunspot_years[0].year}-{sunspot_years[-1].year}  "
    f"({N / 11:.1f} Schwabe periods, {N / 87:.1f} Gleissberg periods, {N / 210:.2f} "
    "Suess/de Vries periods)"
)

This is not a forecasting window — 203 points is not enough to hold out a test span
and still fit an 87-year period, so this section fits the **whole series at once**, the
same thing `seasonal_decompose` did in DAV, using the exact `TrendBasis` and
`SeasonalityBasis` classes from 3.2 with `theta` optimised directly (no `NBeatsBlock`,
no MLP — there is only one series, so there is nothing to generalise across).


In [ ]:
class Decomposition(nn.Module):
    # theta optimised directly against one series -- the MLP in NBeatsBlock exists to
    # produce theta from many different windows; a single 203-point series has no
    # "many windows" to learn that mapping from, so theta itself is the free parameter.
    def __init__(self, bases: list[nn.Module]):
        super().__init__()
        self.bases = nn.ModuleList(bases)
        self._splits = [2 * b.size for b in bases]
        self.theta = nn.Parameter(torch.zeros(1, sum(self._splits)))

    def forward(self):
        parts = torch.split(self.theta, self._splits, dim=1)
        # backcast only
        return sum(b(p)[0] for b, p in zip(self.bases, parts)).squeeze(0)


def fit_decomposition(
    bases: list[nn.Module],
    target: torch.Tensor,
    steps: int = param(1500, test=50),
    lr: float = 0.05,
):
    torch.manual_seed(0)
    model = Decomposition(bases)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        loss = ((model() - target) ** 2).mean()
        loss.backward()
        opt.step()
    with torch.no_grad():
        recon = model()
        r2 = 1 - float(
            ((recon - target) ** 2).sum() / ((target - target.mean()) ** 2).sum()
        )
    return recon, r2


trend = TrendBasis(degree=1, backcast_size=N, forecast_size=1)
schwabe = SeasonalityBasis(harmonics=3, period=11, backcast_size=N, forecast_size=1)
gleissberg = SeasonalityBasis(harmonics=2, period=87, backcast_size=N, forecast_size=1)
# same backcast parameter count as [trend, schwabe, gleissberg] (2+6+4=12), one period only --
# the capacity-matched control from 1_representation_and_head/02_nonlinearity 2.1 and
# 3_scaling_and_search/01_gates_highways_moe 5.6, applied here: if two periods only helped because they are MORE parameters, this ties.
schwabe_wide = SeasonalityBasis(
    harmonics=5, period=11, backcast_size=N, forecast_size=1
)

recon_one, r2_one = fit_decomposition([trend, schwabe], sun)
recon_two, r2_two = fit_decomposition([trend, schwabe, gleissberg], sun)
_, r2_ctrl = fit_decomposition([trend, schwabe_wide], sun)

print(
    f"trend + Schwabe(11yr) only:                R2 = {r2_one:.3f}   (12 backcast params: no)"
)
print(
    f"trend + Schwabe(11yr) + Gleissberg(87yr):   R2 = {r2_two:.3f}   (12 backcast params)"
)
print(
    f"trend + Schwabe(11yr), 5 harmonics [control]: R2 = {r2_ctrl:.3f}   (12 backcast params, "
    "same period)"
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(sunspot_years, sun, lw=0.6, color="0.6", label="yearly mean (scaled)")
ax.plot(
    sunspot_years,
    recon_one.detach(),
    lw=1.2,
    label=f"trend + Schwabe only  (R2={r2_one:.2f})",
)
ax.plot(
    sunspot_years, recon_two.detach(), lw=1.2, label=f"+ Gleissberg  (R2={r2_two:.2f})"
)
ax.legend(fontsize=8)
ax.set_title("fixed-period bases fitted to 203 years of sunspot counts")
plt.tight_layout()

Adding the Gleissberg term is a real, if modest, improvement — and the capacity-matched
control (more harmonics of the *same* 11-year period, same parameter count) gets none
of it: R² sits exactly where the plain Schwabe fit left it. The gain is specifically
from naming the second period, not from having more numbers to fit with — the same
control `3_scaling_and_search/01_gates_highways_moe` 5.6 runs on a completely
different mechanism.

What is left over (R² a little above 0.5, both fits) is the part 203 years cannot buy:
2.3 repeats of Gleissberg is enough to see that the cycle heights rise and fall on
roughly that schedule, not enough to pin its period the way 18.5 Schwabe repeats pin
11 years. And the Suess/de Vries cycle at ~210 years is not even a full repeat in this
record — there is no basis you could add, at any harmonic count, that this dataset
could tell you was right. That is the concrete version of DAV's unfinished suggestion:
not "the decomposition fails," but "here is exactly how much of it 203 years can
support, and here is the line past which you would need more data, not a better model."


## 3.8 How this differs from the reference implementation

Read
[`neuralforecast/models/nbeats.py`](https://github.com/Nixtla/neuralforecast/blob/main/neuralforecast/models/nbeats.py)
next to what you just wrote. `TrendBasis`, `SeasonalityBasis`, `IdentityBasis` and `NBEATSBlock`
are all plain `nn.Module`s taking plain tensors there too, and their stacking loop is the same
five lines. Two deliberate differences:

- **They reverse the window** (`insample_y.flip(dims=(-1,))`) so the most recent step comes first,
  and orient the bases to match. We keep forward time order. Self-consistent either way.
- **Their Fourier frequencies are derived from `forecast_size`**; ours are harmonics of a period
  *you name*. Ours is more readable and connects directly to the FFT bands you read off
  the spectrum, but it requires you to know the period — which the spectrum (3.1) is how you find out.

Their top-level `NBEATS` class subclasses a Lightning `BaseModel` and takes a `windows_batch`
dict, which is the only part that does not transfer. That is the general shape of these libraries:
the ideas are portable, the plumbing is not.

## What to take from this

- **A basis is a constraint on what the model may say**, and constraints are how you get
  interpretability. The stacks separate because the architecture forced them to, not because we
  plotted them apart.
- **Designed features and designed *outputs* are not the same move.** Feed a trend slope in as an
  extra input and the model can ignore it, because a linear layer can already compute one.
  Here the trend is what the model is allowed to predict in, and it cannot be ignored.
- **Read the spectrum before you pick a period.** The seasonality stack is only as useful as the
  period you gave it, and section 3.6 shows what a wrong one buys you.
- **A basis is a bet about structure, not a substitute for data.** Solar power had 103,700
  training windows, ample for the linear layer, and its 144-step Fourier basis matches a
  genuinely periodic process: the interpretable model stayed level with generic N-BEATS and
  both beat linear by ten times the seed noise. Sunspot had 9,147 windows and no 144-step
  period: the interpretable model fell back to linear, while generic, which makes no seasonal
  bet, kept a lead. 3.4 and 3.6 are the same architecture placed on both sides of that bet.
- **Interpretability and accuracy are not always the same trade.** On solar power the model you
  can interrogate scored level with the one you cannot (interpretable 0.2571 vs generic 0.2508,
  a gap of about one seed std). On sunspot the interpretable model lost its edge over linear
  and generic did not (3.6). Which one you ship is still a question about who has to defend
  the forecast, not something the leaderboard settles for you.
- **You still have to beat the cheap baseline.** On solar power's test split, copying
  yesterday (0.2201) beat every trained model in this notebook, though on validation the two
  N-BEATS models edged past it. Whatever the table said, it said it about this series, this
  horizon and this stretch of the year.

## Exercises

**E1 — where did the test score go?** On validation (August to October) both N-BEATS models
edge past seasonal naive; on test (October to December) none does, and every trained model
gets worse while seasonal naive does not (3.4). Score each model and seasonal naive per month
of the test span. Do the trained models lose ground everywhere, or only in some months? Then
set `PATIENCE` to 2 and to 10: do the best epochs and the order on test change?

**E2 — the period is a hyperparameter.** The seasonality stack is fixed to a 144-step (24h)
period. Add a third stack at a week-scale period (1,008 steps) and re-measure. Solar power has
no real week-scale structure (the sun does not know what day it is) — does the extra stack pick
up anything, or does it sit near zero the way a stack tuned to the wrong period should?

**E3 — how much basis is enough?** Sweep `harmonics` over 1, 3, 6, 10, 16 and `degree` over 1, 2,
3. Plot score against the number of basis functions. Where does adding shapes stop helping, and
how does that compare with the band structure the FFT shows on this series?

**E4 — generic versus interpretable, on your own terms.** The generic model has an identity basis:
it can predict any shape. The interpretable one cannot. Measure both as you shrink the training
set (20,000, 5,000, 1,000 windows — pass `n_series` down to `load_solar` to do this cheaply).
Does N-BEATS's edge over linear grow, shrink, or flip as data gets scarcer?

**E5 — search it with ray.** `3_scaling_and_search/04_search` runs exactly this search (`degree`, `harmonics`,
`hidden`, gate on/off) on `solar_10_minutes`. Before opening it, decide which of those axes
you expect to matter, the same way `1_representation_and_head/04_search` asked you to before its own
search ran.

**E6 — the honest comparison.** Add a PatchTST (`transformers.PatchTSTForPrediction`,
Nie et al., 2023) to the table on this series,
with the same seeds and the same early stopping. It has no notion of trend or seasonality and learns
everything from data. On a series this seasonal, does the inductive bias pay?